In [ ]:
import numpy as np
import pandas as pd
import torch
from sklearn.preprocessing import LabelEncoder
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F
import torch.nn as nn
from collections import Counter
torch.__version__

# 5.2 Pytorch???????
## ??
?????,???????????????????????,????????,????????????????????????????????????????,??????????????2????,????csv??,???????,????????Tabular Data??? structured data,?????????????????

???????fastai??????:
https://github.com/fastai/fastai/blob/master/examples/tabular.ipynb
fastai?????


## ?????
??????????,??????csv?????????????,??????????,??????pasdas???????

In [ ]:
#????
df = pd.read_csv('./data/adult.csv')
#salary??????????????
df['salary'].unique()

In [ ]:
#??????
df.head()

In [ ]:
#pandas?describe????????????????,??????
df.describe()

In [ ]:
#?????????
len(df)

???????,????????????,???????????????????
- ???????:??????,????????????????????????????,??????,????????
- ????:?????????,????????????????,???????????????????????,???????????????
- ?????:????????????????,???????????????,???????????????,??????????????????,???????

In [ ]:
#????
result_var = 'salary'
#?????
cat_names = ['workclass', 'education', 'marital-status', 'occupation', 'relationship', 'race','sex','native-country']
#?????
cont_names = ['age', 'fnlwgt', 'education-num','capital-gain','capital-loss','hours-per-week']

??????????,?????????????????????

In [ ]:
for col in df.columns:
    if col in cat_names:
        ccol=Counter(df[col])
        print(col,len(ccol),ccol)
        print("\r\n")

???????????????????,??????,??????????????

In [ ]:
for col in df.columns:
    if col in cat_names:
        df[col].fillna('---')
        df[col] = LabelEncoder().fit_transform(df[col].astype(str))
    if col in cont_names:
        df[col]=df[col].fillna(0)

??????:

???????pandas?fillna???????????????,??????????????????????,???????????? --- ????,???????sklearn?LabelEncoder??????????

???????????????0?????,??????????,????????,????????,?????????,????????

In [ ]:
df.head()

???????????,???????????????,??????????????.

In [ ]:
#??????????
Y = df['salary']
Y_label = LabelEncoder()
Y=Y_label.fit_transform(Y)
Y

In [ ]:
X=df.drop(columns=result_var)
X

??,?????????????,??????????????,????????????????,??????????
## ?????
???pytorch????,?????Dataset????????,????????????

In [ ]:
class tabularDataset(Dataset):
    def __init__(self, X, Y):
        self.x = X.values
        self.y = Y
        
    def __len__(self):
        return len(self.y)
    
    def __getitem__(self, idx):
        return (self.x[idx], self.y[idx])

In [ ]:
train_ds = tabularDataset(X, Y)

?????????????????????

In [ ]:
train_ds[0]

## ????
?????????,??????????????,?????3?????????????

In [ ]:
class tabularModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.lin1 = nn.Linear(14, 500)
        self.lin2 = nn.Linear(500, 100)
        self.lin3 = nn.Linear(100, 2)
        self.bn_in = nn.BatchNorm1d(14)
        self.bn1 = nn.BatchNorm1d(500)
        self.bn2 = nn.BatchNorm1d(100)
        

    def forward(self,x_in):
        #print(x_in.shape)
        x = self.bn_in(x_in)
        x = F.relu(self.lin1(x))
        x = self.bn1(x)
        #print(x)
        
        
        x = F.relu(self.lin2(x))
        x = self.bn2(x)
        #print(x)
        
        x = self.lin3(x)
        x=torch.sigmoid(x)
        return x

????????????????Batch Normalization????????:
??????????????:https://mp.weixin.qq.com/s/FFLQBocTZGqnyN79JbSYcQ

?????????,??????:
![](https://raw.githubusercontent.com/zergtant/pytorch-handbook/master/deephub.jpg)

## ??

In [ ]:
#??????????
DEVICE=torch.device("cpu")
if torch.cuda.is_available():
        DEVICE=torch.device("cuda")
print(DEVICE)

In [ ]:
#????
criterion =nn.CrossEntropyLoss()

In [ ]:
#?????
model = tabularModel().to(DEVICE)
print(model)

In [ ]:
#?????????
rn=torch.rand(3,14).to(DEVICE)
model(rn)

In [ ]:
#???
LEARNING_RATE=0.01
#BS
batch_size = 1024
#???
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)


In [ ]:
#DataLoader????
train_dl = DataLoader(train_ds, batch_size=batch_size,shuffle=True)

??????????????????,????????,???????????

In [ ]:
%%time
model.train()
#??10?
TOTAL_EPOCHS=100
#??????
losses = [];
for epoch in range(TOTAL_EPOCHS):
    for i, (x, y) in enumerate(train_dl):
        x = x.float().to(DEVICE) #?????float??
        y = y.long().to(DEVICE) #???????long??
        #??
        optimizer.zero_grad()
        outputs = model(x)
        #??????
        loss = criterion(outputs, y)
        loss.backward()
        optimizer.step()
        losses.append(loss.cpu().data.item())
    print ('Epoch : %d/%d,   Loss: %.4f'%(epoch+1, TOTAL_EPOCHS, np.mean(losses)))

??????????????????

In [ ]:
model.eval()
correct = 0
total = 0
for i,(x, y) in enumerate(train_dl):
    x = x.float().to(DEVICE)
    y = y.long()
    outputs = model(x).cpu()
    _, predicted = torch.max(outputs.data, 1)
    total += y.size(0)
    correct += (predicted == y).sum()
print('???: %.4f %%' % (100 * correct / total))

??????????